In [3]:
import random
import math
import copy


random: for random neighbor selection and generating random numbers for the acceptance condition.

math: for using the exp (exponential) function in the Metropolis criterion.

copy: for creating deep copies of puzzle states so changes to one state don't affect another.

In [4]:
GOAL = [[1, 2, 3],
        [4, 5, 6],
        [7, 8, 0]]


The desired final state where numbers 1 through 8 are arranged in order, and 0 represents the empty tile

In [5]:
def find_empty(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j

Finds and returns the coordinates (row, column) of the empty tile (0) in the 3×3 puzzle

In [6]:
def manhattan_distance(state):
    distance = 0
    for i in range(3):
        for j in range(3):
            val = state[i][j]
            if val != 0:
                target_i, target_j = (val-1) // 3, (val-1) % 3
                distance += abs(i - target_i) + abs(j - target_j)
    return distance

Manhattan distance heuristic: For each tile (except 0), calculates the distance between its current position and its goal position

Example: Tile 5 in the goal state should be at row 1, column 1 (0-indexed)

Lower values mean the state is closer to the goal. A value of 0 means the puzzle is solved

In [7]:
def get_neighbors(state):
    i, j = find_empty(state)
    neighbors = []
    moves = [(-1,0), (1,0), (0,-1), (0,1)]
    for di, dj in moves:
        ni, nj = i+di, j+dj
        if 0 <= ni < 3 and 0 <= nj < 3:
            new_state = copy.deepcopy(state)
            new_state[i][j], new_state[ni][nj] = new_state[ni][nj], new_state[i][j]
            neighbors.append(new_state)
    return neighbors

Generates all possible neighboring states by swapping the empty tile with its four neighbors (up, down, left, right)

Uses deepcopy to avoid modifying the original state.

In [8]:
def random_neighbor(state):
    i, j = find_empty(state)
    moves = [(-1,0), (1,0), (0,-1), (0,1)]
    random.shuffle(moves)
    for di, dj in moves:
        ni, nj = i+di, j+dj
        if 0 <= ni < 3 and 0 <= nj < 3:
            new_state = copy.deepcopy(state)
            new_state[i][j], new_state[ni][nj] = new_state[ni][nj], new_state[i][j]
            return new_state
    return state

Unlike get_neighbors which returns all neighbors, this function returns only one random neighbor

Uses random.shuffle to randomly check moves until finding a valid one

In [9]:
def print_step(iteration, state, temp, delta, reason):
    """حالت پذیرفته شده در هر تکرار را به همراه جزییات چاپ می‌کند."""
    print(f"--- [ تکرار: {iteration} | دما: {temp:.2f} | دلتا: {delta:+d} ] ---")
    print(f"علت پذیرش: {reason}")
    for row in state:
        print("  " + " ".join(str(cell) if cell != 0 else "_" for cell in row))
    print(f"فاصله منهتن فعلی: {manhattan_distance(state)}")
    print("-" * 40)

In [15]:
def simulated_annealing(initial_state, max_iter=5000, initial_temp=1000, cooling_rate=0.995, temp_min=1e-3):
    current_state = copy.deepcopy(initial_state)
    current_cost = manhattan_distance(current_state)
    best_state = copy.deepcopy(current_state)
    best_cost = current_cost
    temp = initial_temp

    for iteration in range(1, max_iter + 1): # شروع از ۱ برای نمایش بهتر شماره تکرار
        if temp < temp_min:
            break

        # تولید یک همسایه تصادفی
        next_state = random_neighbor(current_state)
        next_cost = manhattan_distance(next_state)

        delta = next_cost - current_cost

        # شرط پذیرش حرکت (معیار metropolis)
        accepted = False
        reason = ""
        
        if delta < 0:
            accepted = True
            reason = "✅ حرکت رو به بهبود (هزینه کمتر)"
        else:
            prob = math.exp(-delta / temp)
            if random.random() < prob:
                accepted = True
                reason = f"🎲 حرکت بدتر (پذیرش بر اساس شانس با احتمال {prob:.3f})"

        if accepted:
            current_state = next_state
            current_cost = next_cost

            # فراخوانی تابع جدید برای نمایش وضعیت پذیرفته شده
            print_step(iteration, current_state, temp, delta, reason)

            # به‌روزرسانی بهترین حالت
            if current_cost < best_cost:
                best_state = copy.deepcopy(current_state)
                best_cost = current_cost
                # اگر به جواب کامل رسیدیم، پایان
                if best_cost == 0:
                    break

        # کاهش دما
        temp = initial_temp / math.log(1 + (cooling_rate * iteration) + 1e-9)

    return best_state, best_cost, iteration

In [16]:
def print_puzzle(state):
    for row in state:
        print(" ".join(str(cell) if cell != 0 else " " for cell in row))
    print("فاصله منهتن:", manhattan_distance(state))

In [18]:
if __name__ == "__main__":
    # یک حالت اولیه با کمی فاصله از هدف برای دیدن مراحل پذیرش
    initial = [[8, 7, 2],
               [4, 6, 5],
               [0, 3, 1]]

    print("حالت اولیه:")
    print_puzzle(initial)
    print("\nدر حال اجرای الگوریتم شبیه‌سازی ذوب فلزات...\n")

    # پارامترها کمی تغییر کرده تا خروجی ترمینال خیلی طولانی نشود
    final_state, final_cost, steps = simulated_annealing(initial, max_iter=10000, initial_temp=100, cooling_rate=0.99)

    print("\nحالت نهایی (بهترین حالت یافت شده):")
    print_puzzle(final_state)
    print(f"تعداد تکرار انجام شده: {steps}")
    if final_cost == 0:
        print("✅ مسئله با موفقیت حل شد!")
    else:
        print(f"⚠️ جواب کامل پیدا نشد. بهترین فاصله منهتن = {final_cost}")

حالت اولیه:
8 7 2
4 6 5
  3 1
فاصله منهتن: 16

در حال اجرای الگوریتم شبیه‌سازی ذوب فلزات...

--- [ تکرار: 1 | دما: 100.00 | دلتا: +1 ] ---
علت پذیرش: 🎲 حرکت بدتر (پذیرش بر اساس شانس با احتمال 0.990)
  8 7 2
  4 6 5
  3 _ 1
فاصله منهتن فعلی: 17
----------------------------------------
--- [ تکرار: 2 | دما: 145.32 | دلتا: -1 ] ---
علت پذیرش: ✅ حرکت رو به بهبود (هزینه کمتر)
  8 7 2
  4 6 5
  _ 3 1
فاصله منهتن فعلی: 16
----------------------------------------
--- [ تکرار: 3 | دما: 91.58 | دلتا: +1 ] ---
علت پذیرش: 🎲 حرکت بدتر (پذیرش بر اساس شانس با احتمال 0.989)
  8 7 2
  _ 6 5
  4 3 1
فاصله منهتن فعلی: 17
----------------------------------------
--- [ تکرار: 4 | دما: 72.53 | دلتا: -1 ] ---
علت پذیرش: ✅ حرکت رو به بهبود (هزینه کمتر)
  8 7 2
  4 6 5
  _ 3 1
فاصله منهتن فعلی: 16
----------------------------------------
--- [ تکرار: 5 | دما: 62.45 | دلتا: +1 ] ---
علت پذیرش: 🎲 حرکت بدتر (پذیرش بر اساس شانس با احتمال 0.984)
  8 7 2
  4 6 5
  3 _ 1
فاصله منهتن فعلی: 17
-------------------------